In [ ]:
!pip install wordcloud unidecode nltk


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
from tqdm import tqdm
import re
from unidecode import unidecode
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
tqdm.pandas()

In [ ]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
df=pd.read_csv('https://breathecode.herokuapp.com/asset/internal-link?id=932&path=url_spam.csv')
df.head()

,url,is_spam
0,https://briefingday.us8.list-manage.com/unsubs...,True
1,https://www.hvper.com/,True
2,https://briefingday.com/m/v4n3i4f3,True
3,https://briefingday.com/n/20200618/m#commentform,False
4,https://briefingday.com/fan,True


In [ ]:
def preprocess(text):
    text = text.lower()

    # extraer y conservar partes de la url si existe
    url_text = ''
    m = re.search(r"(https?://\S+|www\.\S+)", text)
    if m:
        u = m.group(0)
        u = re.sub(r"https?://", '', u)
        u = re.sub(r"^www\.", '', u)
        parts = re.split(r"[\./\?=&#_-]+", u)
        url_text = ' '.join([p for p in parts if p])

    # Cotempla las posibles formas en las que puede aparecer una url
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)
    # Eliminación de las menciones
    text = re.sub(r'@\w+', '', text)
    # Eliminación de hashtags
    text = re.sub(r'#\w+', '', text)
    # Solo texto
    text = re.sub(r'[^\w\s]|[\d]', '', text)
    # Eliminación de los espacios adicionales
    text = re.sub(r'\s+', ' ', text).strip()
    # Tokenización del texto
    tokens = word_tokenize(text)
    # Eliminación de los acentos
    tokens = [unidecode(token) for token in tokens]
    # Eliminacion de stopwords
    stop_words = stopwords.words('spanish')
    stop_words.append('espana')
    tokens = [token for token in tokens if token not in stop_words]
    # Lematización de palabras
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    result = ' '.join(tokens)
    if url_text:
        return f"{result} {url_text}".strip()
    return result

In [ ]:
df['url_prepro'] = df['url'].progress_apply(preprocess)

AttributeError: 'Series' object has no attribute '_is_builtin_func'

In [ ]:
# mostrar algunas entradas y estadística de longitud
print(df['url_prepro'].head(10))


0          briefingday us8 list manage com unsubscribe
1                                            hvper com
2                           briefingday com m v4n3i4f3
3             briefingday com n 20200618 m commentform
4                                  briefingday com fan
5    brookings edu interactives reopening america a...
6    reuters com investigates special report health...
7    theatlantic com magazine archive 2020 07 super...
8    vox com 2020 6 17 21294680 john bolton book ex...
9    theguardian com travel 2020 jun 18 end of tour...
Name: url_prepro, dtype: object


In [ ]:
df.shape

(2999, 3)

In [ ]:
df['url_prepro'].duplicated().sum()

np.int64(634)

In [ ]:
# eliminar duplicados y mostrar el nuevo número de urls
df.drop_duplicates(subset='url_prepro', inplace=True)
print(f'El dataset se compone de un total de {df.shape[0]} urls.')

El dataset se compone de un total de 2365 urls.


In [ ]:
# Comprobación de valores faltantes
df['url_prepro'].isna().sum()

np.int64(0)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df.url_prepro, df.is_spam, test_size=0.2, random_state=42)

In [ ]:
# Crear una instancia del LabelEncoder
label_encoder = LabelEncoder()

# Ajustar y transformar la columna 'is_spam' con el LabelEncoder
y_train_label = label_encoder.fit_transform(y_train)
y_test_label = label_encoder.transform(y_test)

In [ ]:
# 1. Unir todos los textos de la columna 'url_prepro' en un solo bloque
# Eliminamos posibles valores nulos con dropna() para evitar errores
todo_el_texto = " ".join(df['url_prepro'].dropna())

# 2. Generar la nube de palabras global
# He aumentado el max_words a 50 para que la imagen sea más vistosa, 
# pero puedes volver a 10 si prefieres solo lo más extremo.
wordcloud = WordCloud(
    background_color="white", 
    max_words=50, 
    contour_color="steelblue", 
    collocations=False, # Evita que se repitan palabras combinadas
    width=800, 
    height=400
).generate(todo_el_texto)

# 3. Mostrar la imagen final
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.title("Palabras más destacadas en las URLs procesadas")
plt.axis("off")
plt.show()

NameError: name 'df' is not defined

In [ ]:
# Convertimos las palabras a números
vectorizer = TfidfVectorizer(min_df=0.001)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
X_train_vec.toarray().shape

(1892, 1871)

In [ ]:
clf = LogisticRegression(class_weight="balanced").fit(X_train_vec.toarray(), y_train_label)

In [ ]:
y_pred = clf.predict(X_test_vec.toarray())
# label_encoder.classes_ puede contener booleans; classification_report espera strings
print(classification_report(y_test_label, y_pred, target_names=[str(c) for c in label_encoder.classes_]))

              precision    recall  f1-score   support

       False       0.95      0.89      0.92       420
        True       0.42      0.64      0.51        53

    accuracy                           0.86       473
   macro avg       0.69      0.76      0.71       473
weighted avg       0.89      0.86      0.87       473

